In [0]:
# Demo Setup: Delta Table Statistics and Optimizations
# Creates tables to demonstrate DESCRIBE DETAIL, data skipping,
# OPTIMIZE, and liquid clustering.

import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "workspace"
schema = f"demo_delta_stats_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

In [0]:
# --- Generate 5000 rows of sales data ---
random.seed(42)

regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Food', 'Home', 'Sports']
channels = ['online', 'store', 'partner']

rows = []
for i in range(5000):
    sale_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    rows.append((
        i + 1,
        sale_date,
        random.choice(regions),
        random.choice(categories),
        random.choice(channels),
        round(random.uniform(10, 500), 2)
    ))

schema_def = StructType([
    StructField("event_id", IntegerType(), False),
    StructField("sale_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("product_category", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("amount", DoubleType(), False)
])

df = spark.createDataFrame(rows, schema=schema_def)

# --- Create unclustered table (single write) ---
spark.sql("DROP TABLE IF EXISTS sales_events")
df.write.saveAsTable("sales_events")

In [0]:
# --- Insert additional small batches to create multiple files ---
# This simulates how streaming/frequent appends create many small files.
for batch in range(8):
    batch_rows = []
    for i in range(200):
        sale_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
        batch_rows.append((
            5001 + batch * 200 + i,
            sale_date,
            random.choice(regions),
            random.choice(categories),
            random.choice(channels),
            round(random.uniform(10, 500), 2)
        ))
    batch_df = spark.createDataFrame(batch_rows, schema=schema_def)
    batch_df.write.mode("append").saveAsTable("sales_events")

total_rows = spark.sql("SELECT COUNT(*) FROM sales_events").collect()[0][0]

In [0]:
# --- Create clustered version of the same data ---
spark.sql("DROP TABLE IF EXISTS sales_events_clustered")

# Read all data from the unclustered table
all_data = spark.table("sales_events")

# Write with liquid clustering on sale_date and region
all_data.write.clusterBy("sale_date", "region").saveAsTable("sales_events_clustered")

In [0]:
# --- Create a small dimension table ---
spark.sql("DROP TABLE IF EXISTS product_lookup")

spark.sql("""
  CREATE TABLE product_lookup (
    product_category STRING,
    category_manager STRING,
    profit_margin_pct DOUBLE
  )
""")

spark.sql("""
  INSERT INTO product_lookup VALUES
    ('Electronics', 'Alice Chen', 0.22),
    ('Clothing', 'Bob Park', 0.45),
    ('Food', 'Carol Lin', 0.15),
    ('Home', 'Dave Wu', 0.35),
    ('Sports', 'Eva Kim', 0.30)
""")

In [0]:
# --- Summary ---
print(f"SETUP COMPLETE")
print(f"")
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Tables created:")
print(f"sales_events           - {total_rows} rows, NO clustering (multiple files)")
print(f"sales_events_clustered - {all_data.count()} rows, CLUSTER BY (sale_date, region)")
print(f"product_lookup         - 5 rows (dimension table)")